## 1. Import Libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Read CSV Files

In [2]:
employment_by_industry = pd.read_csv("Data/employment_by_industry.csv")

## 3. Show First 5 Rows Of Datasets

In [3]:
employment_by_industry.head()

,year,sex,industry1,industry2,employment_status,employed
0,2010,male,manufacturing,manufacturing,employers,10800
1,2010,male,manufacturing,manufacturing,employees,170500
2,2010,male,manufacturing,manufacturing,own account workers,4400
3,2010,male,manufacturing,manufacturing,contributing family workers,400
4,2010,male,construction,construction,employers,11000


## 4. Data Cleaning

#### 4.A. Code to make a copy of ges_degree_details dataframe for cleaning

In [4]:
employment_by_industry_clean = employment_by_industry.copy()
print("Shape:", employment_by_industry_clean.shape)

Shape: (1920, 6)


### 4.B. Dataset Summary before cleaning 

In [5]:
column_summary = pd.DataFrame({
    "data_type": employment_by_industry_clean.dtypes.astype(str),
    "missing_count": (employment_by_industry_clean.isna().sum()),
    "missing_percentage": (employment_by_industry_clean.isna().mean() * 100).round(2),
    "unique_count": (employment_by_industry_clean.nunique(dropna = False)
    )
})

display(column_summary)

print("Empty Rows:", employment_by_industry_clean.isna().all(axis=1).sum())
print("Exact duplicate rows:", employment_by_industry_clean.duplicated().sum())
print("Duplicate identifier records:", employment_by_industry_clean.duplicated(subset =["year", "sex", "industry1", "industry2", "employment_status"]).sum())

,data_type,missing_count,missing_percentage,unique_count
year,int64,0,0.0,16
sex,str,0,0.0,2
industry1,str,0,0.0,4
industry2,str,0,0.0,15
employment_status,str,0,0.0,4
employed,int64,0,0.0,513


Empty Rows: 0
Exact duplicate rows: 0
Duplicate identifier records: 0


### 4.C. Checking for Placeholder Values 

In [6]:
placeholder_values = {
    "",
    "na",
    "n/a",
    "null",
    "none",
    "unknown"
    }

placeholder_report = []

for column in [
    "sex",
    "industry1",
    "industry2",
    "employment_status"
]:
    normalised_values = (
        employment_by_industry_clean[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    count = normalised_values.isin(placeholder_values).sum()

    placeholder_report.append({
        "Column": column,
        "Placeholder Count": count
    })

display(pd.DataFrame(placeholder_report))



,Column,Placeholder Count
0,sex,0
1,industry1,0
2,industry2,0
3,employment_status,0


### 4.D. Covert Placeholders Values to NaN

In [7]:
for column in [
    "sex",
    "industry1",
    "industry2",
    "employment_status"
]:
    normalised_values = (
        employment_by_industry_clean[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    placeholder_mask = normalised_values.isin(placeholder_values)

    employment_by_industry_clean.loc[placeholder_mask, column] = np.nan


print("Missing values after replacing placeholders:")
display(employment_by_industry_clean.isna().sum())


Missing values after replacing placeholders:


year                 0
sex                  0
industry1            0
industry2            0
employment_status    0
employed             0
dtype: int64

### 4.E. Check for Duplicate Records 

In [8]:
duplicate_rows = employment_by_industry_clean.duplicated().sum()
print("Verifying duplicate rows before removal:")

Verifying duplicate rows before removal:


### 4.F. Removing Duplicates Records (if any)

In [9]:
rows_before = len(employment_by_industry_clean)
employment_by_industry_clean = (
    employment_by_industry_clean.drop_duplicates()
)

### resetting the index after dropping duplicates to ensure a clean DataFrame
employment_by_industry_clean.reset_index(drop=True, inplace=True)

rows_removed = (
    rows_before - len(employment_by_industry_clean)
)

print("Rows removed:", rows_removed)


Rows removed: 0


### 4.G. Validating the Year Column then Finding and Removing Invalid Years 

In [10]:
### check whether every year is numeric 
year_numeric_test = pd.to_numeric(
    employment_by_industry_clean["year"],
    errors="coerce"
)


### finding invalid years
invalid_year_mask = (
    year_numeric_test.isna() & employment_by_industry_clean["year"].notna()
)

print("Rows containing invalid year values:", invalid_year_mask.sum())

display(
    employment_by_industry_clean.loc[
        invalid_year_mask
    ]
)


### Removing Invalid Years 
rows_before = len(employment_by_industry_clean)

employment_by_industry_clean = (employment_by_industry_clean.loc[
    ~invalid_year_mask
        ].copy())

rows_removed = (rows_before - len(employment_by_industry_clean))

print("Rows removed due to invalid years:", rows_removed)



Rows containing invalid year values: 0


,year,sex,industry1,industry2,employment_status,employed


Rows removed due to invalid years: 0


### 4.H. Converting Year to Integer and Checking Year Distribution

In [11]:
### Converting Year to Integer

employment_by_industry_clean["year"] = (
    pd.to_numeric(employment_by_industry_clean["year"],
        errors = "coerce").astype("Int64")
)

print(employment_by_industry_clean["year"].dtype)


### Check year Distribution

display(
    employment_by_industry_clean["year"].value_counts(dropna=False).sort_index()
)

Int64


year
2010    120
2011    120
2012    120
2013    120
2014    120
2015    120
2016    120
2017    120
2018    120
2019    120
2020    120
2021    120
2022    120
2023    120
2024    120
2025    120
Name: count, dtype: Int64

### 4.I. Checking Consistency for Gender, Industries and Employment Status

In [12]:
display(employment_by_industry_clean["sex"].value_counts(dropna=False))

display(employment_by_industry_clean["industry1"].value_counts(dropna=False))

display(employment_by_industry_clean["industry2"].value_counts(dropna=False))

display(employment_by_industry_clean["employment_status"].value_counts(dropna=False))


sex
male      960
female    960
Name: count, dtype: int64

industry1
services         1536
manufacturing     128
construction      128
others            128
Name: count, dtype: int64

industry2
manufacturing                                    128
construction                                     128
wholesale and retail trade                       128
transportation and storage                       128
accommodation and food services                  128
information and communications                   128
financial and insurance services                 128
real estate services                             128
professional services                            128
administrative and support services              128
public administration and education              128
health and social services                       128
arts, entertainment and recreation               128
other community, social and personal services    128
others                                           128
Name: count, dtype: int64

employment_status
employers                      480
employees                      480
own account workers            480
contributing family workers    480
Name: count, dtype: int64

### 4.J. Standardising Text Columns 

In [13]:
text_columns = [
    "sex",
    "industry1",
    "industry2",
    "employment_status"
]

for column in text_columns:

    employment_by_industry_clean[column] = (
        employment_by_industry_clean[column]
        .astype("string")
        .str.strip()
        .str.title()
    )


### 4.K. Verifying Standardised Categories

In [14]:
print("Unique values in Sex:")
display(sorted(employment_by_industry_clean["sex"].dropna().unique()))

print("Unique values in Industry1:")
display(sorted(employment_by_industry_clean["industry1"].dropna().unique()))

print("Unique values in Industry2:")
display(sorted(employment_by_industry_clean["industry2"].dropna().unique()))

print("Unique values in Employment Status:")
display(sorted(employment_by_industry_clean["employment_status"].dropna().unique()))



Unique values in Sex:


['Female', 'Male']

Unique values in Industry1:


['Construction', 'Manufacturing', 'Others', 'Services']

Unique values in Industry2:


['Accommodation And Food Services',
 'Administrative And Support Services',
 'Arts, Entertainment And Recreation',
 'Construction',
 'Financial And Insurance Services',
 'Health And Social Services',
 'Information And Communications',
 'Manufacturing',
 'Other Community, Social And Personal Services',
 'Others',
 'Professional Services',
 'Public Administration And Education',
 'Real Estate Services',
 'Transportation And Storage',
 'Wholesale And Retail Trade']

Unique values in Employment Status:


['Contributing Family Workers',
 'Employees',
 'Employers',
 'Own Account Workers']

### 4.L. Validating Data Types and Employment Values 

In [15]:
employment_by_industry_clean.dtypes

negative_values =  (
    employment_by_industry_clean["employed"] < 0 
).sum()

print("Negative employment values:", negative_values)

Negative employment values: 0


### 4.M. Summary of data after cleaning 

In [16]:
print("Final Shape:")
print(employment_by_industry_clean.shape)

print("\nMissing Values:")
display(employment_by_industry_clean.isna().sum())

print("\nDuplicate Rows:")
print(employment_by_industry_clean.duplicated().sum())

print("\nData Types:")
display(employment_by_industry_clean.dtypes)

Final Shape:
(1920, 6)

Missing Values:


year                 0
sex                  0
industry1            0
industry2            0
employment_status    0
employed             0
dtype: int64


Duplicate Rows:
0

Data Types:


year                  Int64
sex                  string
industry1            string
industry2            string
employment_status    string
employed              int64
dtype: object